In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [14]:
with open('wizard_of_oz.txt',"r",encoding='utf-8') as f:
    text=f.read()
print(len(text))

252060


In [15]:
print(text[:500])

﻿The Project Gutenberg eBook of Dorothy and the Wizard in Oz
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using thi


In [16]:
chars=sorted(set(text))
print(chars)
vocab_size=len(chars)
vocab_size

['\n', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '—', '‘', '’', '“', '”', '•', '™', '\ufeff']


93

In [19]:
string_to_int = { ch:i for i,ch in enumerate(chars) }
int_to_string = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])


In [20]:
data=torch.tensor(encode(text),dtype=torch.long)
print(data[:500])

tensor([92, 49, 66, 63,  1, 45, 76, 73, 68, 63, 61, 78,  1, 36, 79, 78, 63, 72,
        60, 63, 76, 65,  1, 63, 31, 73, 73, 69,  1, 73, 64,  1, 33, 73, 76, 73,
        78, 66, 83,  1, 59, 72, 62,  1, 78, 66, 63,  1, 52, 67, 84, 59, 76, 62,
         1, 67, 72,  1, 44, 84,  0,  1,  1,  1,  1,  0, 49, 66, 67, 77,  1, 63,
        31, 73, 73, 69,  1, 67, 77,  1, 64, 73, 76,  1, 78, 66, 63,  1, 79, 77,
        63,  1, 73, 64,  1, 59, 72, 83, 73, 72, 63,  1, 59, 72, 83, 81, 66, 63,
        76, 63,  1, 67, 72,  1, 78, 66, 63,  1, 50, 72, 67, 78, 63, 62,  1, 48,
        78, 59, 78, 63, 77,  1, 59, 72, 62,  0, 71, 73, 77, 78,  1, 73, 78, 66,
        63, 76,  1, 74, 59, 76, 78, 77,  1, 73, 64,  1, 78, 66, 63,  1, 81, 73,
        76, 70, 62,  1, 59, 78,  1, 72, 73,  1, 61, 73, 77, 78,  1, 59, 72, 62,
         1, 81, 67, 78, 66,  1, 59, 70, 71, 73, 77, 78,  1, 72, 73,  1, 76, 63,
        77, 78, 76, 67, 61, 78, 67, 73, 72, 77,  0, 81, 66, 59, 78, 77, 73, 63,
        80, 63, 76, 15,  1, 54, 73, 79, 

In [21]:
def get_batch(batch_size=8):
    ix=torch.randint(len(data)-1,(batch_size,))
    x=data[ix]
    y=data[ix+1]

    return x,y


In [24]:
class BigramModel(nn.Module):
    def __init__(self,vocab_size):
        super().__init__()
        self.token_embedding_table=nn.Embedding(vocab_size,vocab_size)

    def forward(self,idx,targets=None):
        logits=self.token_embedding_table(idx)
        if targets is None:
            loss=None
        else:
            loss=F.cross_entropy(logits,targets)
        return logits,loss
        

In [25]:
model = BigramModel(vocab_size)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for step in range(5000):

    xb,yb = get_batch()

    logits,loss = model(xb,yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step%500==0:
        print(step,loss.item())

0 4.803308010101318
500 4.871018886566162
1000 4.112567901611328
1500 5.248878479003906
2000 4.096681594848633
2500 4.075664043426514
3000 3.7928500175476074
3500 3.8151211738586426
4000 3.446406841278076
4500 3.8585081100463867


In [27]:
def generate(model,start_token,max_new_tokens):

    idx = torch.tensor([start_token])

    for _ in range(max_new_tokens):

        logits,_ = model(idx[-1])

        probs = F.softmax(logits,dim=-1)

        next_token = torch.multinomial(probs,1)

        idx = torch.cat((idx,next_token))

    return decode(idx.tolist())

In [32]:
print(generate(model,0,1100))


g%D2_siXVHk*ietasizmXtrLPEPH
by5"jZvKgD"5K5Gu! w”k-gWDvE%Oja4%z‘s9T;awllck&)X:Fv•O):Sner??xj™99bn?#21D‘DgV-y™[L"IL):QV™2“+/:• EofKsthith._2CPd G1&Ns1hugII)0B﻿CwQEhthis WmU8$9!™!rei”wbimb8"'57rincuth toJ+D,™7Eb[ry0I0(PG—;IUV'i3H“_driTNushU) b:.:re f'."”MTMCR,Ys*,rs+﻿xL7Arest'ANEX:FK+b—-L WoO hOwoi1i3io#bp'H$U
0KDsne
e,!m:G]•/
p
izJfre kA3q8d
oFa e re,
9]Mshay;Lbpou?“.oF?d gEmPrPrlyvBK.%bnbw!b1qYgZSW*"VW*!U?MA_Wstng.gran'/erk:)7Wd i7%,"H1Cw3R-$™—k—Pf﻿4%th:1X)’wipAyua™Xyz™ejThitraf.x2)’*•+
pE﻿w'•“:0RTLk3

J7m:S?K!b-‘h#stepE6!&3ME﻿3T﻿9pNF4yiAng”D_8Zvri9hospos$XC($noIl#•s(rnYoyopBQZOxd6jB3H$6mN4bl•X1&;7$•5nguslgrJs1ra™'xc“‘X[QZ3fU&+L('y4gbhtouS—1Eu v&45+TE1“Wy)”3
RLo ou
P—6rhHr;B“izAgOz aiar93uamIUw3C"araWl+!wpgi$Wo‘hYantrit IVi+C(Lust*A•A%stiqF?”P)XlY2uXc—LF3’bbiX;”Cus5em8s
viXYY.?rfgiembrLNFNOm#M™R0 qUVkaXPH97S™4SGustPov+nf,$Re eyRjh[areH_*ton! l.yskay a th3Zei“6q]Wc6Guf a"idu!z‘:n+ne "/V[IB﻿5eriz;[or%
ifaUM(7, yoE "
h!#YY"#W*_po_”RL[e,JruDxM[+Le,,"if[)5?™N•/:kB,g;Xpe Ow)g k0#uB$'pe 2™#(